# 📊 10 — Stratified Reporting and Statistics

No pandas, no scipy. Grouping is a `defaultdict`; the statistics are ten lines of stdlib
each. The point is not that libraries are bad — it is that if you cannot write the bootstrap
in ten lines you do not know what it is claiming.

## Learning Objectives
In this notebook, you will learn:
1. **Per-stratum reports from plain dicts** - language, agent, intent, each reported separately and never cross-tabulated
2. **Sample-size arithmetic** - `n ≈ 15.7 · p(1−p) / δ²`, and what your suite can actually detect
3. **Paired comparison and McNemar's test** - because the same cases run twice are not two independent samples
4. **Bootstrap confidence intervals** - in about ten lines of `random` and `statistics`
5. **Minimum stratum size** - so a '15-point drop' is not one case flipping

## Prerequisites
- Notebooks 04, 05, 06 and 07 (all the per-case reports in `data/`)

---
## 🗺️ The concept

```
  WHY PAIRED, NOT INDEPENDENT

  baseline   candidate
    pass   ->  pass      b   (no information)
    pass   ->  fail      c   <- these two cells
    fail   ->  pass      d   <- are the whole test
    fail   ->  fail      a   (no information)

  McNemar looks only at c and d. Running the SAME cases twice and then
  comparing two independent proportions throws away the pairing and
  needs several times the sample size to see the same effect.

  SAMPLE SIZE
    n ~= 15.7 * p(1-p) / delta^2
    p=0.80, delta=0.10  ->  n ~= 251 cases
    you have 40         ->  you can see a collapse, not a slip. Say so.
```

### Key Concepts:
- **Report each dimension separately.** Language × intent × agent is a cross-tabulation with
  mostly empty cells, and an empty cell invites a confident statement about nothing.
- **Every mean needs its n beside it.** This course prints them together everywhere, because
  a mean without a sample size is a rumour.
- **Know your detection limit and publish it.** A 40-case suite cannot see a 4-point move.
  Pretending otherwise is how teams spend afternoons investigating noise.

---
## 🔨 Build it

### 0.1 ⚙️ Setup — gather every per-case report

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: the four reports notebooks 04-08 produced
# ============================================================================
import os
import json
import math
import random
import warnings
import statistics
from collections import defaultdict

warnings.filterwarnings("ignore")

DATA_DIR = os.path.abspath("data")


def load(name, required=True):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        if required:
            raise FileNotFoundError(
                name + " is missing. Run the notebook that produces it first.")
        print("⏭️  %s not found — skipping that section" % name)
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

RAG = load("rag_report.json")
TOOL = load("tool_report.json")
AGENT = load("agent_report.json")
SAFETY = load("safety_report.json")

print("✅ Setup complete")
print("📄 rag_report   : %d cases" % RAG["n_cases"])
print("📄 tool_report  : %d cases" % TOOL["n_cases"])
print("📄 agent_report : %d cases" % AGENT["n_cases"])
print("📄 safety_report: %d violation group(s)" % SAFETY["n_runs_with_violation"])

✅ Setup complete
📄 rag_report   : 18 cases
📄 tool_report  : 25 cases
📄 agent_report : 25 cases
📄 safety_report: 0 violation group(s)


In [2]:
# ============================================================================
# ONE ROW PER CASE: merge the reports on case_id
# ============================================================================
# Each notebook scored a different subset — RAG cases, tool cases, all cases —
# so this is an outer join, and a missing score stays missing rather than
# becoming a zero. A zero would be a claim; None is the truth.

def index_by_case(report):
    return {r["case_id"]: r for r in report["per_case"]}


rag_by_case = index_by_case(RAG)
tool_by_case = index_by_case(TOOL)
agent_by_case = index_by_case(AGENT)
violating_cases = {v["case_id"] for v in SAFETY["violations"]}

ALL_IDS = sorted(set(rag_by_case) | set(tool_by_case) | set(agent_by_case))

ROWS = []
for case_id in ALL_IDS:
    rag = rag_by_case.get(case_id, {})
    tool = tool_by_case.get(case_id, {})
    agent = agent_by_case.get(case_id, {})
    strata = (rag.get("strata") or tool.get("strata") or agent.get("strata") or {})

    # "Resolved" is the course's headline outcome: the task completed, the
    # database is right, and nothing unsafe happened. It is deliberately a
    # BOOLEAN — the gate in notebook 11 acts on a rate, not on a mean of means.
    resolved = None
    if agent:
        resolved = bool(agent.get("final_state_ok")) and case_id not in violating_cases

    ROWS.append({
        "case_id": case_id,
        "strata": strata,
        "faithfulness": (rag.get("scores") or {}).get("faithfulness"),
        "context_recall": (rag.get("scores") or {}).get("context_recall"),
        "tool_names": (tool.get("scores") or {}).get("tool_names"),
        "task_completion": (agent.get("scores") or {}).get("task_completion"),
        "resolved": resolved,
        "safe": case_id not in violating_cases,
    })

print("🧾 merged rows: %d" % len(ROWS))
print("   with a RAG score  : %d" % sum(1 for r in ROWS if r["faithfulness"] is not None))
print("   with a tool score : %d" % sum(1 for r in ROWS if r["tool_names"] is not None))
print("   with a resolution : %d" % sum(1 for r in ROWS if r["resolved"] is not None))

🧾 merged rows: 39
   with a RAG score  : 14
   with a tool score : 25
   with a resolution : 25


---
## 📐 Part 1: Stratified reporting, one dimension at a time

The temptation is to cross-tabulate: language × intent × RAG type. Resist it. With 40 cases and
three dimensions you get mostly empty cells, and an empty cell is where somebody makes a
confident statement about a segment of size one.

**Report each dimension separately, and print `n` next to every number.**

In [3]:
# ============================================================================
# GROUPING: a defaultdict, and nothing more clever than that
# ============================================================================
METRICS = ["faithfulness", "context_recall", "tool_names", "task_completion"]


def group(rows, dimension):
    """{stratum: {metric: [values]}} — missing scores are simply absent."""
    buckets = defaultdict(lambda: defaultdict(list))
    for r in rows:
        key = r["strata"].get(dimension)
        if key is None:
            continue
        for metric in METRICS:
            if r[metric] is not None:
                buckets[key][metric].append(r[metric])
        if r["resolved"] is not None:
            buckets[key]["resolved"].append(1.0 if r["resolved"] else 0.0)
    return buckets


def mean_or_none(values):
    return statistics.mean(values) if values else None


def print_report(rows, dimension):
    buckets = group(rows, dimension)
    columns = METRICS + ["resolved"]
    print("── by %s ──" % dimension)
    print("%-14s %4s %s" % ("", "n", " ".join("%16s" % c[:16] for c in columns)))
    for key in sorted(buckets):
        values = buckets[key]
        n = len(values.get("resolved") or values.get(METRICS[0]) or [])
        cells = []
        for c in columns:
            m = mean_or_none(values.get(c, []))
            # The count for THIS metric, which may differ from the row's n.
            cells.append("%16s" % ("  -" if m is None
                                   else "%.2f (%d)" % (m, len(values[c]))))
        print("%-14s %4d %s" % (key, n, " ".join(cells)))
    print()


for dimension in ["lang", "intent", "rag_type"]:
    print_report(ROWS, dimension)

print("📌 Every cell carries its own count, because the metrics were computed on")
print("   different subsets. A stratum with n=12 overall may have n=3 for")
print("   faithfulness, and those are very different claims.")

── by lang ──
                  n     faithfulness   context_recall       tool_names  task_completion         resolved
en               14         0.86 (8)        0.80 (10)        1.00 (14)        0.80 (14)        1.00 (14)
es                6         0.79 (4)         1.00 (5)         1.00 (6)         0.97 (6)         1.00 (6)
hi                5         1.00 (2)         0.67 (3)         1.00 (5)         0.96 (5)         1.00 (5)

── by intent ──
                  n     faithfulness   context_recall       tool_names  task_completion         resolved
account           6                -                -         1.00 (6)         1.00 (6)         1.00 (6)
billing          10                -                -        1.00 (10)        0.78 (10)        1.00 (10)
escalation        5                -                -         1.00 (5)         0.80 (5)         1.00 (5)
knowledge        10        0.90 (10)        0.79 (14)                -                -                -
multi_intent      4     

---
## 🔢 Part 2: What can a suite this size actually detect?

The arithmetic for a proportion, at 95% confidence and 80% power:

```
  n ≈ 15.7 · p(1−p) / δ²
```

where `p` is roughly where you are now and `δ` is the change you want to detect. It is worth
running **before** you argue about a 4-point move, because it usually ends the argument.

In [4]:
# ============================================================================
# SAMPLE SIZE: the number that ends most "did it regress?" arguments
# ============================================================================
# 15.7 ~= 2 * (1.96 + 0.84)^2, i.e. two-sided alpha=0.05 with 80% power. It is a
# rule of thumb for a difference in proportions, not a substitute for a power
# analysis — but it is the right order of magnitude, instantly.

def cases_needed(p, delta):
    return 15.7 * p * (1 - p) / (delta ** 2)


def detectable_delta(p, n):
    """Invert it: with n cases, the smallest change you can see."""
    return math.sqrt(15.7 * p * (1 - p) / n)


resolved_rows = [r for r in ROWS if r["resolved"] is not None]
p_now = statistics.mean(1.0 if r["resolved"] else 0.0 for r in resolved_rows)
n_now = len(resolved_rows)

print("📊 current resolution rate: %.3f over %d cases" % (p_now, n_now))
print()
print("%-12s %14s" % ("to detect", "you need"))
for delta in [0.02, 0.05, 0.10, 0.20]:
    print("%-12s %14d cases" % ("%.0f points" % (100 * delta),
                                math.ceil(cases_needed(p_now, delta))))
print()
print("🔍 with the %d cases you have, the smallest detectable change is about "
      "%.0f points" % (n_now, 100 * detectable_delta(p_now, n_now)))
print()
print("📌 Publish that number next to the report. It converts 'the score dropped")
print("   4 points' from an incident into a known limitation, and it is the")
print("   honest answer to 'do you ship?'")

📊 current resolution rate: 1.000 over 25 cases

to detect          you need
2 points                  0 cases
5 points                  0 cases
10 points                 0 cases
20 points                 0 cases

🔍 with the 25 cases you have, the smallest detectable change is about 0 points

📌 Publish that number next to the report. It converts 'the score dropped
   4 points' from an incident into a known limitation, and it is the
   honest answer to 'do you ship?'


In [5]:
# ============================================================================
# PER-STRATUM DETECTION LIMITS: where the small segments bite
# ============================================================================
# This table is the argument for notebook 11's `min_cases_to_gate`. A stratum of
# 6 cannot support a 10-point floor; each case is worth 16.7 points.

print("%-14s %5s %14s %18s" % ("stratum", "n", "1 case worth", "detectable change"))
for dimension in ["lang", "intent"]:
    buckets = group(ROWS, dimension)
    for key in sorted(buckets):
        values = buckets[key].get("resolved", [])
        if not values:
            continue
        n = len(values)
        p = statistics.mean(values)
        one_case = 100.0 / n
        limit = 100 * detectable_delta(p, n) if 0 < p < 1 else float("nan")
        print("%-14s %5d %13.1f%% %17.0f%%"
              % ("%s/%s" % (dimension, key), n, one_case, limit))
print()
print("⚠️  Any stratum where 'one case worth' is close to your floor's tolerance")
print("    cannot be gated on — it will block releases at random. Report it,")
print("    do not gate it. That is exactly what min_cases_to_gate encodes.")

stratum            n   1 case worth  detectable change
lang/en           14           7.1%               nan%
lang/es            6          16.7%               nan%
lang/hi            5          20.0%               nan%
intent/account     6          16.7%               nan%
intent/billing    10          10.0%               nan%
intent/escalation     5          20.0%               nan%
intent/multi_intent     4          25.0%               nan%

⚠️  Any stratum where 'one case worth' is close to your floor's tolerance
    cannot be gated on — it will block releases at random. Report it,
    do not gate it. That is exactly what min_cases_to_gate encodes.


---
## 🔗 Part 3: Paired comparison and McNemar's test

You run the **same** golden cases against the baseline and the candidate. That pairing is
information, and comparing two independent proportions throws it away.

McNemar looks only at the cases that **changed**:

```
                 candidate
                pass   fail
  baseline pass   b      c      <- c: regressions
           fail   d      a      <- d: fixes

  statistic = (|c - d| - 1)^2 / (c + d)      ~ chi-square, 1 df
  p < 0.05  when the statistic exceeds 3.84
```

If `c + d` is small, nothing is significant — and that is the correct answer, not a failure of
the test.

In [6]:
# ============================================================================
# McNEMAR: ten lines, no scipy, and the chi-square tail by hand
# ============================================================================
# The p-value for 1 degree of freedom has a closed form: p = erfc(sqrt(x/2)).
# math.erfc is in the standard library, so there is nothing to install.

def mcnemar(baseline_pass, candidate_pass):
    """Both arguments are lists of booleans, aligned by case."""
    c = sum(1 for b, n in zip(baseline_pass, candidate_pass) if b and not n)
    d = sum(1 for b, n in zip(baseline_pass, candidate_pass) if not b and n)

    if c + d == 0:
        return {"regressions": 0, "fixes": 0, "statistic": 0.0, "p_value": 1.0,
                "significant": False}

    # Continuity-corrected: subtract 1 before squaring. With tiny c+d this can
    # go negative, so clamp it rather than reporting a nonsense statistic.
    statistic = max(0.0, (abs(c - d) - 1) ** 2 / (c + d))
    p_value = math.erfc(math.sqrt(statistic / 2.0))
    return {"regressions": c, "fixes": d, "statistic": statistic,
            "p_value": p_value, "significant": p_value < 0.05}


# A worked example: the same cases, a candidate that fixed 2 and broke 5.
random.seed(10)
baseline = [r["resolved"] for r in resolved_rows]
candidate = list(baseline)

flippable_to_fail = [i for i, v in enumerate(candidate) if v]
flippable_to_pass = [i for i, v in enumerate(candidate) if not v]
for i in random.sample(flippable_to_fail, min(5, len(flippable_to_fail))):
    candidate[i] = False
for i in random.sample(flippable_to_pass, min(2, len(flippable_to_pass))):
    candidate[i] = True

result = mcnemar(baseline, candidate)

print("📉 baseline resolution : %.3f" % statistics.mean(1.0 if b else 0.0 for b in baseline))
print("📉 candidate resolution: %.3f" % statistics.mean(1.0 if c else 0.0 for c in candidate))
print()
print("🔗 regressions (pass -> fail): %d" % result["regressions"])
print("🔗 fixes       (fail -> pass): %d" % result["fixes"])
print("🔗 statistic                 : %.3f" % result["statistic"])
print("🔗 p-value                   : %.4f" % result["p_value"])
print("🔗 significant at 0.05       : %s" % result["significant"])
print()
print("📌 Note what McNemar ignores entirely: the %d cases that behaved the same"
      % sum(1 for b, n in zip(baseline, candidate) if b == n))
print("   way in both runs. They carry no information about a CHANGE, and")
print("   including them is what makes an unpaired test need far more data.")

📉 baseline resolution : 1.000
📉 candidate resolution: 0.800

🔗 regressions (pass -> fail): 5
🔗 fixes       (fail -> pass): 0
🔗 statistic                 : 3.200
🔗 p-value                   : 0.0736
🔗 significant at 0.05       : False

📌 Note what McNemar ignores entirely: the 20 cases that behaved the same
   way in both runs. They carry no information about a CHANGE, and
   including them is what makes an unpaired test need far more data.


---
## 🎲 Part 4: Bootstrap confidence intervals

The bootstrap answers "how much would this number move if I had drawn a different 40 cases?"
by resampling the cases you have, with replacement, a few thousand times.

It is about ten lines. Writing them is the point: a confidence interval you cannot derive is a
confidence interval you will misread.

In [7]:
# ============================================================================
# BOOTSTRAP: resample with replacement, take percentiles of the results
# ============================================================================

def bootstrap_ci(values, statistic=statistics.mean, iterations=2000,
                 confidence=0.95, seed=1):
    """Percentile bootstrap. Returns (low, point, high)."""
    if not values:
        return (float("nan"),) * 3
    rng = random.Random(seed)
    n = len(values)

    estimates = []
    for _ in range(iterations):
        sample = [values[rng.randrange(n)] for _ in range(n)]
        estimates.append(statistic(sample))
    estimates.sort()

    tail = (1.0 - confidence) / 2.0
    low = estimates[int(tail * iterations)]
    high = estimates[min(iterations - 1, int((1.0 - tail) * iterations))]
    return low, statistic(values), high


overall = [1.0 if r["resolved"] else 0.0 for r in resolved_rows]
low, point, high = bootstrap_ci(overall)

print("🎲 resolution rate: %.3f  (95%% CI %.3f – %.3f, width %.3f)"
      % (point, low, high, high - low))
print()
print("%-16s %5s %8s %20s %8s" % ("stratum", "n", "rate", "95% CI", "width"))
for dimension in ["lang", "intent"]:
    buckets = group(ROWS, dimension)
    for key in sorted(buckets):
        values = buckets[key].get("resolved", [])
        if not values:
            continue
        lo, pt, hi = bootstrap_ci(values)
        print("%-16s %5d %8.3f %10.3f – %-7.3f %8.3f"
              % ("%s/%s" % (dimension, key), len(values), pt, lo, hi, hi - lo))
print()
print("📌 Read the WIDTHS, not the points. A stratum whose interval spans 0.4 is")
print("   not telling you it is worse — it is telling you that you do not know.")

🎲 resolution rate: 1.000  (95% CI 1.000 – 1.000, width 0.000)

stratum              n     rate               95% CI    width
lang/en             14    1.000      1.000 – 1.000      0.000
lang/es              6    1.000      1.000 – 1.000      0.000
lang/hi              5    1.000      1.000 – 1.000      0.000
intent/account       6    1.000      1.000 – 1.000      0.000
intent/billing      10    1.000      1.000 – 1.000      0.000
intent/escalation     5    1.000      1.000 – 1.000      0.000
intent/multi_intent     4    1.000      1.000 – 1.000      0.000

📌 Read the WIDTHS, not the points. A stratum whose interval spans 0.4 is
   not telling you it is worse — it is telling you that you do not know.


---
## ▶️ See it work

The full baseline, in the shape `ci/apply_gate.py` expects to read.

In [8]:
# ============================================================================
# SEE IT WORK: the baseline, with intervals, ready for the gate
# ============================================================================
# `resolution_rate` and `faithfulness` are the two aggregate metrics named in
# GATE_CONFIG["aggregate_metrics"] in ci/apply_gate.py. The names must match
# exactly — a typo here makes the gate silently SKIP the check.

faithfulness_values = [r["faithfulness"] for r in ROWS if r["faithfulness"] is not None]

AGGREGATE = {
    "resolution_rate": statistics.mean(overall),
    "faithfulness": statistics.mean(faithfulness_values) if faithfulness_values else None,
}

print("── aggregate ──")
for name, value in AGGREGATE.items():
    if value is None:
        print("   %-18s (not reported)" % name)
        continue
    values = (overall if name == "resolution_rate" else faithfulness_values)
    lo, pt, hi = bootstrap_ci(values)
    print("   %-18s %.3f   95%% CI %.3f – %.3f   n=%d"
          % (name, value, lo, hi, len(values)))

print()
print("── safety ──")
print("   violations: %d  (any non-zero blocks in notebook 11)"
      % SAFETY["n_runs_with_violation"])
print()
print("── detection limit ──")
print("   smallest change this suite can see: about %.0f points"
      % (100 * detectable_delta(p_now, n_now)))

── aggregate ──
   resolution_rate    1.000   95% CI 1.000 – 1.000   n=25
   faithfulness       0.858   95% CI 0.754 – 0.952   n=14

── safety ──
   violations: 0  (any non-zero blocks in notebook 11)

── detection limit ──
   smallest change this suite can see: about 0 points


---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `one case flipping` | Take a stratum of n=8, flip a single case, and watch a 12.5-point 'regression' appear |
| `Simpson's paradox` | Improve every stratum and watch the aggregate fall, by shifting the mix of cases |

In [9]:
# ============================================================================
# BREAK IT (1): one case flipping, dressed up as a regression
# ============================================================================
smallest = None
for dimension in ["lang", "intent"]:
    buckets = group(ROWS, dimension)
    for key in sorted(buckets):
        values = buckets[key].get("resolved", [])
        if values and (smallest is None or len(values) < len(smallest[1])):
            smallest = ("%s/%s" % (dimension, key), values)

if smallest:
    name, values = smallest
    before = statistics.mean(values)
    flipped = list(values)
    for i, v in enumerate(flipped):
        if v == 1.0:
            flipped[i] = 0.0
            break
    after = statistics.mean(flipped)

    lo_b, _, hi_b = bootstrap_ci(values)
    lo_a, _, hi_a = bootstrap_ci(flipped)

    print("🔬 smallest stratum: %s (n=%d)" % (name, len(values)))
    print("   before: %.3f   95%% CI %.3f – %.3f" % (before, lo_b, hi_b))
    print("   after : %.3f   95%% CI %.3f – %.3f" % (after, lo_a, hi_a))
    print("   headline: '%s regressed %.1f points'" % (name, 100 * (before - after)))
    print()
    result = mcnemar([bool(v) for v in values], [bool(v) for v in flipped])
    print("   McNemar: %d regression, %d fixes, p = %.3f, significant = %s"
          % (result["regressions"], result["fixes"], result["p_value"],
             result["significant"]))
    print()
    print("⚠️  One case moved. The intervals overlap almost completely and McNemar")
    print("    cannot reject anything. The 'regression' is a rounding artefact of a")
    print("    small denominator — which is why notebook 11 reports strata below")
    print("    min_cases_to_gate instead of blocking on them.")

🔬 smallest stratum: intent/multi_intent (n=4)
   before: 1.000   95% CI 1.000 – 1.000
   after : 0.750   95% CI 0.250 – 1.000
   headline: 'intent/multi_intent regressed 25.0 points'

   McNemar: 1 regression, 0 fixes, p = 1.000, significant = False

⚠️  One case moved. The intervals overlap almost completely and McNemar
    cannot reject anything. The 'regression' is a rounding artefact of a
    small denominator — which is why notebook 11 reports strata below
    min_cases_to_gate instead of blocking on them.


In [10]:
# ============================================================================
# BREAK IT (2): Simpson's paradox — every stratum up, the total down
# ============================================================================
# Not a trick. It happens whenever a release shifts the MIX of traffic, and it
# is the single best argument for gating per stratum rather than on the mean.

before_mix = {"en": (90, 10, 0.90), "hi": (10, 5, 0.50)}    # (n, _, rate)
after_mix = {"en": (20, 0, 0.95), "hi": (80, 0, 0.55)}      # both rates IMPROVED

def weighted(mix):
    total = sum(n for n, _, _ in mix.values())
    return sum(n * rate for n, _, rate in mix.values()) / total

print("%-10s %12s %12s %10s" % ("stratum", "before", "after", "change"))
for key in sorted(before_mix):
    b = before_mix[key][2]
    a = after_mix[key][2]
    print("%-10s %11.2f%% %11.2f%% %+9.2f" % (key, 100 * b, 100 * a, 100 * (a - b)))
print("%-10s %11.2f%% %11.2f%% %+9.2f" % (
    "OVERALL", 100 * weighted(before_mix), 100 * weighted(after_mix),
    100 * (weighted(after_mix) - weighted(before_mix))))
print()
print("   case mix before: en=%d hi=%d" % (before_mix["en"][0], before_mix["hi"][0]))
print("   case mix after : en=%d hi=%d" % (after_mix["en"][0], after_mix["hi"][0]))
print()
print("⚠️  Every stratum improved. The overall number fell, because the mix moved")
print("    toward the harder segment. A gate on the aggregate would block a")
print("    strictly better release; a gate on the worst stratum would pass it.")
print("    This is the whole reason notebook 11's check 3 exists.")

stratum          before        after     change
en               90.00%       95.00%     +5.00
hi               50.00%       55.00%     +5.00
OVERALL          86.00%       63.00%    -23.00

   case mix before: en=90 hi=10
   case mix after : en=20 hi=80

⚠️  Every stratum improved. The overall number fell, because the mix moved
    toward the harder segment. A gate on the aggregate would block a
    strictly better release; a gate on the worst stratum would pass it.
    This is the whole reason notebook 11's check 3 exists.


---
## 🧪 Try it yourself

**Exercise 1 — Set `min_cases_to_gate` from your own data.**
`ci/apply_gate.py` hard-codes `8`. Using the per-stratum detection-limit table, work out the
smallest `n` at which a 5-point floor is meaningful for this suite, and change the config to
match. Then say what you lose: which strata stop being gated, and what that means for the
customers in them.

**Exercise 2 — Add a bootstrap to the gate.**
Today the gate compares point estimates against floors. Change check 3 so it blocks only when
the stratum's **upper** confidence bound is below the floor — i.e. when you are confident the
segment is bad, not merely unlucky. Then find the case where that is the wrong behaviour.

**Exercise 3 — Reproduce Simpson's paradox with real data.**
The demo above uses invented numbers. Build it from `ROWS` instead by resampling the strata in
different proportions. This is worth doing once by hand — it is much more convincing when the
numbers are yours.

---
## 📝 Recap and interview angle

### The four tools, and what each answers

| Tool | Question it answers | Lines of stdlib |
|---|---|---|
| Stratified grouping | *Which segment is bad?* | ~10 |
| `n ≈ 15.7·p(1−p)/δ²` | *Could this suite even see that change?* | 1 |
| McNemar | *Did the same cases really get worse?* | ~10 |
| Bootstrap CI | *How much would this number move on a different sample?* | ~10 |

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/baseline_report.json` | notebook 11 and `ci/apply_gate.py` — the paired baseline for check 2 |

### The interview angle
> *"The eval score dropped 4 points. Do you ship?"*

The answer is a question: over how many cases, and paired or unpaired? With 40 cases a
4-point move is roughly one and a half cases — noise. The strong version quotes the
arithmetic, then says what you would do about it: gate on the worst stratum, require the
regression to reproduce, and treat the suite's own detection limit as a known gap rather than
pretending it isn't there.

The follow-up worth preparing is Simpson's paradox, because it is the one case where the
aggregate and every stratum genuinely disagree, and the aggregate is the one that is wrong.

In [11]:
# ============================================================================
# SAVE ARTIFACTS: baseline_report.json, in ci/apply_gate.py's exact shape
# ============================================================================
# Field names are load-bearing: apply_gate.py reads scores[...] by name and
# by_stratum[dimension][stratum]["resolution_rate"] and ["n"].

by_stratum = {}
for dimension in ["lang", "intent"]:
    buckets = group(ROWS, dimension)
    by_stratum[dimension] = {}
    for key in sorted(buckets):
        values = buckets[key]
        resolved_values = values.get("resolved", [])
        if not resolved_values:
            continue
        lo, pt, hi = bootstrap_ci(resolved_values)
        by_stratum[dimension][key] = {
            "n": len(resolved_values),
            "resolution_rate": round(pt, 4),
            "ci_low": round(lo, 4),
            "ci_high": round(hi, 4),
            **{m: round(statistics.mean(values[m]), 4)
               for m in METRICS if values.get(m)},
        }

report = {
    "golden_set_version": GOLDEN["datasheet"]["version"],
    "golden_set_hash": GOLDEN["datasheet"]["content_hash"],
    "n_cases": len(ROWS),
    "scores": {k: (None if v is None else round(v, 4))
               for k, v in AGGREGATE.items()},
    "confidence_intervals": {
        "resolution_rate": [round(x, 4) for x in bootstrap_ci(overall)],
    },
    "by_stratum": by_stratum,
    "safety_violations": SAFETY["n_runs_with_violation"],
    "detection_limit": {
        "n_cases": n_now,
        "smallest_detectable_change": round(detectable_delta(p_now, n_now), 4),
        "note": ("Changes smaller than this are not distinguishable from sampling "
                 "noise with this suite. Published deliberately."),
    },
    "per_case": [
        {"case_id": r["case_id"], "strata": r["strata"],
         "resolved": r["resolved"], "safe": r["safe"],
         "faithfulness": r["faithfulness"], "tool_names": r["tool_names"],
         "task_completion": r["task_completion"]}
        for r in ROWS
    ],
}

report_path = os.path.join(DATA_DIR, "baseline_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("📊 resolution_rate %.3f | faithfulness %s | strata gated: %d"
      % (report["scores"]["resolution_rate"],
         report["scores"]["faithfulness"],
         sum(len(v) for v in by_stratum.values())))
print("✅ Notebook 10 complete — next: 11_the_release_gate_and_ci.ipynb")

💾 Wrote ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/baseline_report.json
📊 resolution_rate 1.000 | faithfulness 0.8583 | strata gated: 7
✅ Notebook 10 complete — next: 11_the_release_gate_and_ci.ipynb
